In [6]:
import os
import json
import zipfile
import numpy as np
import copy
import requests
import io
import shutil

import pandas as pd
import geopandas as gpd

import shapely.wkt as wkt
from shapely.geometry import LineString
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import BoundaryNorm

from math import radians, cos, sin, asin, sqrt

In [ ]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"DC")
os.makedirs(DIR, exist_ok=True)

dc_shape="./Neighborhood_Clusters.geojson"
dc_shapefile_url="https://hub.arcgis.com/api/v3/datasets/f6c703ebe2534fc3800609a07bad8f5b_17/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1"

taxi_orig_dir=os.path.join(DIR,"taxi")
bike_orig_dir=os.path.join(DIR,"bike")

# Download

Neighborhood Clusters https://hub.arcgis.com/datasets/DCGIS::neighborhood-clusters  
Taxi 2025 https://dcgov.app.box.com/v/TaxiTrips2025  
Bike 2025 https://s3.amazonaws.com/capitalbikeshare-data/index.html  

## 1 Shapefile

In [8]:
if not os.path.exists(dc_shape):
    response = requests.get(dc_shapefile_url)
    with open(dc_shape, "w", encoding="utf-8") as file:
        file.write(response.text)

## 2 Taxi

The DC taxi trips for 2025 are published as a single zip on Box
(https://dcgov.app.box.com/v/TaxiTrips2025). The Box "Download" button hits a
public shared-file endpoint that 302-redirects to the actual zip on
`boxcloud.com`. We resolve it directly with the `shared_name` / `file_id`
extracted from the folder's `Box.postStreamData`, then extract the archive
(`OpenDataDC_Taxi_2025.zip`, ~525 MB) into `taxi/`.

In [9]:
DOWNLOAD=False
os.makedirs(taxi_orig_dir, exist_ok=True)

# Box public shared-file download endpoint (equivalent to clicking "Download").
shared_name = "jzqft9mx6o5ywceem9v2t1tt3zale0zy"
file_id = "f_2094775022176"
taxi_url = (
    "https://dcgov.app.box.com/index.php?rm=box_download_shared_file"
    f"&shared_name={shared_name}&file_id={file_id}"
)
zip_filepath = os.path.join(taxi_orig_dir, "OpenDataDC_Taxi_2025.zip")

if DOWNLOAD:
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(taxi_url, headers=headers, stream=True, timeout=120)
        if response.status_code == 200:
            with open(zip_filepath, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192 * 1024):  # 8MB
                    if chunk:
                        f.write(chunk)

            with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
                zip_ref.extractall(taxi_orig_dir)
            os.remove(zip_filepath)
        else:
            print(f"Error: {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")

## 3 Bike

In [ ]:
DOWNLOAD=False
year = 2025
base_url_template = "https://s3.amazonaws.com/capitalbikeshare-data/{year}{month:02d}-capitalbikeshare-tripdata.zip"
os.makedirs(bike_orig_dir, exist_ok=True)
if DOWNLOAD:
    for month in range(1, 13):
        url = base_url_template.format(year=year, month=month)
        zip_filename = f"{year}{month:02d}-capitalbikeshare-tripdata.zip"
        zip_filepath = os.path.join(bike_orig_dir, zip_filename)
        try:
            response = requests.get(url, stream=True, timeout=120)

            if response.status_code == 200:
                with open(zip_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 1024):  # 8MB
                        if chunk:
                            f.write(chunk)

                with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                    zip_ref.extractall(bike_orig_dir)
                os.remove(zip_filepath)

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")